# 🧩 Structured Output

## Use case

Les LLMs génèrent du texte libre, utile pour un humain, difficile à exploiter dans un programme.

Le **structured output** contraint le modèle à répondre dans un format défini (JSON, objet typé) validé par un schéma. Deux cas d'usage fondamentaux :

- **Génération**: produire un objet structuré depuis un prompt
- **Extraction**: extraire des données structurées depuis du texte non structuré

Fil rouge de ce notebook : la **user story** agile.

## Stack

- **OpenAI SDK** (`openai`): `beta.chat.completions.parse` + intégration Pydantic
- **Pydantic**: validation de schéma Python-first
- **LiteLLM**: approche provider-agnostic en fin de notebook

## What's next

- `02-augmentation/function-calling`: structured output appliqué aux outils
- `03-agentique/use-cases`: structured output pour orchestrer des agents (Reflection pattern)

## Setup

**En local**
1. Copier `.env.example` en `.env` à la racine du repo
2. Renseigner `OPENAI_API_KEY`
3. Lancer le notebook avec ton environnement uv ou jupyter habituel

**Sur Google Colab**
1. Ouvrir les Secrets (icône 🔑 dans le panneau gauche)
2. Ajouter un secret `OPENAI_API_KEY` avec ta clé
3. Activer l'accès au secret pour ce notebook

In [ ]:
%pip install openai pydantic litellm python-dotenv -q

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv(override=True)
except ImportError:
    pass  # Colab — variables disponibles via les secrets

assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY manquante — voir .env.example (local) ou Secrets (Colab)"
)

In [ ]:
from openai import OpenAI

client = OpenAI()

## Pourquoi structured output ?

L'approche naïve : demander au modèle de répondre en JSON dans le prompt, puis `json.loads()`.

Les problèmes en pratique :
- **Réponse invalide**: le modèle entoure le JSON de markdown (` ```json ``` `), ajoute du texte avant/après
- **Clés hallucinées**: le modèle invente des champs non demandés, ou en omet
- **Types incohérents**: un champ attendu en liste retourné en string
- **Parsing fragile**: un seul caractère invalide fait crasher `json.loads`

Le structured output résout ça en contraignant la génération au niveau du modèle, pas en post-processing. Le schéma est garanti, et non pas espéré.

## Génération

Produire une user story structurée depuis un prompt.

On définit le schéma attendu avec Pydantic, puis on passe ce schéma à `beta.chat.completions.parse`.
Le modèle est contraint à respecter la structure, `parsed` est directement utilisable.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class UserStory(BaseModel):
    title: str
    user_story: str = Field(description="Format: En tant que... je veux... afin de...")
    acceptance_criteria: List[str]

In [ ]:
generation_response = client.beta.chat.completions.parse(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Tu es un expert en rédaction de user stories agiles."},
        {"role": "user", "content": "Génère une user story pour une fonctionnalité de connexion via Google."},
    ],
    response_format=UserStory,
)

# Réponse brute — le modèle ne fait pas de magie, c'est du JSON contraint
print("Réponse brute:", generation_response)

In [ ]:
# Output parsé et typé — directement utilisable
user_story = generation_response.choices[0].message.parsed
print("Title:", user_story.title)
print("User story:", user_story.user_story)
print("Acceptance criteria:", user_story.acceptance_criteria)

## Extraction

Extraire les éléments structurés d'une user story rédigée en langage naturel.

Même mécanique, prompt différent, le modèle lit le texte et remplit le schéma.
C'est le cas d'usage le plus fréquent en production : parser des emails, des tickets, des documents.

In [ ]:
class ExtractedUserStory(BaseModel):
    actor: str = Field(description="Qui fait l'action")
    action: str = Field(description="Ce que l'acteur veut faire")
    benefit: str = Field(description="La valeur métier")
    acceptance_criteria: List[str]

In [ ]:
raw_user_story = """
    En tant qu'utilisateur je voudrais pouvoir me connecter avec mon compte Google
    pour ne pas avoir à créer un nouveau mot de passe et accéder plus rapidement
    à l'application. La fonctionnalité devra proposer un bouton Google sur la page
    de login, rediriger vers l'auth Google et créer le compte automatiquement si
    c'est la première connexion.
"""

extraction_response = client.beta.chat.completions.parse(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Tu es un expert en méthodes agiles. Extrais les éléments structurés de cette user story."},
        {"role": "user", "content": raw_user_story},
    ],
    response_format=ExtractedUserStory,
)

# Réponse brute
print("Réponse brute:", extraction_response)

In [ ]:
extracted = extraction_response.choices[0].message.parsed
print("Actor:", extracted.actor)
print("Action:", extracted.action)
print("Benefit:", extracted.benefit)
print("Acceptance criteria:", extracted.acceptance_criteria)

## Vers une approche provider-agnostic avec LiteLLM

LiteLLM supporte le structured output via le paramètre `response_format`, même interface,
même schéma Pydantic, provider configurable en tête de notebook.

> **Note** — le support varie selon les providers. OpenAI et Anthropic (claude-3+) supportent
> nativement le JSON schema contraint. Vérifier la [compatibilité LiteLLM](https://docs.litellm.ai/docs/completion/json_mode) pour les autres.

In [ ]:
# 👇 Changer ici pour switcher de provider
MODEL = "openai/gpt-4.1-mini"
# MODEL = "anthropic/claude-haiku-4-5"

### Génération

In [ ]:
import json
from litellm import completion

response = completion(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Tu es un expert en rédaction de user stories agiles."},
        {"role": "user", "content": "Génère une user story pour une fonctionnalité de connexion via Google."},
    ],
    response_format=UserStory,
)

# Output parsé depuis le JSON retourné
result = UserStory(**json.loads(response.choices[0].message.content))
print(result)
print(f"\nTokens — prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens}")

### Extraction

In [ ]:
response = completion(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Tu es un expert en méthodes agiles. Extrais les éléments structurés de cette user story."},
        {"role": "user", "content": raw_user_story},
    ],
    response_format=ExtractedUserStory,
)

result = ExtractedUserStory(**json.loads(response.choices[0].message.content))
print(result)